# 使用PyTorch构建回归MLP

> 本笔记本是 [01-顺序API构建回归MLP.ipynb](./01-顺序API构建回归MLP.ipynb) 的 **PyTorch 等价版本**，
> 原版使用 TensorFlow/Keras Sequential API，本版使用 PyTorch nn.Sequential 实现相同功能。

本教程演示如何使用PyTorch构建多层感知机(MLP)来解决回归问题。

## 学习目标

1. 掌握PyTorch nn.Sequential的基本用法
2. 理解回归任务的模型构建流程
3. 学会数据预处理和特征标准化
4. 掌握PyTorch训练循环的编写方法

## 数据集介绍

使用California Housing数据集，该数据集包含1990年加州各区块的房价信息：
- 8个特征：收入中位数、房龄、房间数等
- 目标变量：房价中位数（单位：10万美元）

## 1. 环境配置

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from torch.utils.data import DataLoader, TensorDataset

# 设置随机种子确保结果可复现
RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)
torch.manual_seed(RANDOM_SEED)

# 检测并选择计算设备
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

print(f"PyTorch版本: {torch.__version__}")
print(f"计算设备: {device}")

## 2. 数据加载与探索

In [ ]:
# 加载California Housing数据集
housing = fetch_california_housing()

# 查看数据集基本信息
print("数据集描述:")
print("="*50)
print(f"特征数量: {housing.data.shape[1]}")
print(f"样本数量: {housing.data.shape[0]}")
print(f"\n特征名称: {housing.feature_names}")
print("\n目标变量: 房价中位数 (单位: 10万美元)")

In [ ]:
# 创建DataFrame便于数据探索
df = pd.DataFrame(housing.data, columns=housing.feature_names)
df['MedHouseVal'] = housing.target

print("数据统计信息:")
df.describe()

## 3. 数据预处理

### 数据集划分策略

将数据划分为三部分：
- **训练集 (60%)**: 用于模型参数学习
- **验证集 (20%)**: 用于超参数调优和早停
- **测试集 (20%)**: 用于最终模型评估

In [ ]:
# 第一次划分：分离出测试集
X_train_full, X_test, y_train_full, y_test = train_test_split(
    housing.data, housing.target,
    test_size=0.2,
    random_state=RANDOM_SEED
)

# 第二次划分：从剩余数据中分离出验证集
X_train, X_valid, y_train, y_valid = train_test_split(
    X_train_full, y_train_full,
    test_size=0.25,  # 0.25 * 0.8 = 0.2
    random_state=RANDOM_SEED
)

print(f"训练集大小: {X_train.shape[0]}")
print(f"验证集大小: {X_valid.shape[0]}")
print(f"测试集大小: {X_test.shape[0]}")

### 特征标准化

神经网络对输入特征的尺度敏感，需要进行标准化处理：
$$x_{\text{scaled}} = \frac{x - \mu}{\sigma}$$

**注意**: 标准化参数(均值和标准差)只能从训练集计算，然后应用到验证集和测试集。

In [ ]:
# 创建标准化器
scaler = StandardScaler()

# 在训练集上拟合并转换
X_train = scaler.fit_transform(X_train)

# 在验证集和测试集上只进行转换（使用训练集的参数）
X_valid = scaler.transform(X_valid)
X_test = scaler.transform(X_test)

print("标准化后的训练数据统计:")
print(f"均值: {X_train.mean(axis=0).round(2)}")
print(f"标准差: {X_train.std(axis=0).round(2)}")

### 转换为PyTorch张量与DataLoader

PyTorch使用TensorDataset和DataLoader来管理数据批次，
这相当于Keras中 `model.fit()` 的 `batch_size` 参数。

In [ ]:
# 将NumPy数组转换为PyTorch张量
# 特征使用float32，目标变量需要reshape为(N, 1)以匹配模型输出形状
X_train_t = torch.tensor(X_train, dtype=torch.float32)
y_train_t = torch.tensor(y_train, dtype=torch.float32).unsqueeze(1)  # (N,) -> (N, 1)

X_valid_t = torch.tensor(X_valid, dtype=torch.float32)
y_valid_t = torch.tensor(y_valid, dtype=torch.float32).unsqueeze(1)

X_test_t = torch.tensor(X_test, dtype=torch.float32)
y_test_t = torch.tensor(y_test, dtype=torch.float32).unsqueeze(1)

# 创建TensorDataset和DataLoader
BATCH_SIZE = 32

train_dataset = TensorDataset(X_train_t, y_train_t)
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)

valid_dataset = TensorDataset(X_valid_t, y_valid_t)
valid_loader = DataLoader(valid_dataset, batch_size=BATCH_SIZE, shuffle=False)

test_dataset = TensorDataset(X_test_t, y_test_t)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)

print(f"训练DataLoader批次数: {len(train_loader)}")
print(f"验证DataLoader批次数: {len(valid_loader)}")
print(f"测试DataLoader批次数: {len(test_loader)}")

## 4. 构建MLP模型

### nn.Sequential介绍

nn.Sequential是PyTorch中最简单的模型构建方式，适用于层的线性堆叠，
类似于Keras的Sequential API。

### 模型架构

```
输入层 (8个特征)
    ↓
隐藏层1 (30个神经元, ReLU激活)
    ↓
输出层 (1个神经元, 线性激活)
```

In [ ]:
# 方式1：使用nn.Sequential构建模型（等价于Keras Sequential）
input_dim = X_train.shape[1]  # 8个特征

model = nn.Sequential(
    nn.Linear(input_dim, 30),  # 输入层 -> 隐藏层1: 8 -> 30
    nn.ReLU(),                 # ReLU激活函数
    nn.Linear(30, 1)           # 隐藏层1 -> 输出层: 30 -> 1（线性输出）
)

# 将模型移至计算设备
model = model.to(device)

# 查看模型结构
print(model)
print(f"\n模型参数总数: {sum(p.numel() for p in model.parameters())}")

In [ ]:
# 方式2：使用nn.Module子类定义（更灵活的等效写法）
# class RegressionMLP(nn.Module):
#     def __init__(self, input_dim):
#         super().__init__()
#         self.hidden1 = nn.Linear(input_dim, 30)
#         self.relu = nn.ReLU()
#         self.output = nn.Linear(30, 1)
#
#     def forward(self, x):
#         x = self.relu(self.hidden1(x))
#         return self.output(x)
#
# model = RegressionMLP(input_dim=8).to(device)

## 5. 定义损失函数和优化器

PyTorch中需要分别定义损失函数和优化器，
这相当于Keras中 `model.compile()` 的 `loss` 和 `optimizer` 参数：
- **损失函数**: 回归任务使用均方误差(MSE)
- **优化器**: SGD（随机梯度下降）
- **评估指标**: MAE更具可解释性（PyTorch中需手动计算）

In [ ]:
# 定义损失函数和优化器
criterion = nn.MSELoss()                          # MSE损失函数
optimizer = torch.optim.SGD(model.parameters(), lr=1e-2)  # SGD优化器，学习率0.01

print(f"损失函数: {criterion}")
print(f"优化器: {optimizer}")

## 6. 训练模型

PyTorch需要手动编写训练循环，而Keras使用 `model.fit()` 一行搞定。
虽然代码更多，但提供了更细粒度的控制。

In [ ]:
def train_epoch(model, dataloader, criterion, optimizer, device):
    """
    训练模型一个epoch
    Train the model for one epoch.

    Parameters:
    -----------
    model : nn.Module
        待训练的PyTorch模型 / PyTorch model to train
    dataloader : DataLoader
        训练数据加载器 / Training data loader
    criterion : nn.Module
        损失函数 / Loss function
    optimizer : torch.optim.Optimizer
        优化器 / Optimizer
    device : torch.device
        计算设备 (CPU/GPU) / Compute device

    Returns:
    --------
    float : 该epoch的平均损失 / Average loss for the epoch
    """
    model.train()
    total_loss = 0.0
    for X_batch, y_batch in dataloader:
        X_batch, y_batch = X_batch.to(device), y_batch.to(device)
        optimizer.zero_grad()
        outputs = model(X_batch)
        loss = criterion(outputs, y_batch)
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * X_batch.size(0)
    return total_loss / len(dataloader.dataset)


def evaluate(model, dataloader, criterion, device):
    """
    评估模型性能
    Evaluate the model on a dataset.

    Parameters:
    -----------
    model : nn.Module
        待评估的PyTorch模型 / PyTorch model to evaluate
    dataloader : DataLoader
        评估数据加载器 / Evaluation data loader
    criterion : nn.Module
        损失函数 / Loss function
    device : torch.device
        计算设备 (CPU/GPU) / Compute device

    Returns:
    --------
    tuple : (平均MSE损失, 平均MAE) / (average MSE loss, average MAE)
    """
    model.eval()
    total_loss = 0.0
    total_mae = 0.0
    with torch.no_grad():
        for X_batch, y_batch in dataloader:
            X_batch, y_batch = X_batch.to(device), y_batch.to(device)
            outputs = model(X_batch)
            total_loss += criterion(outputs, y_batch).item() * X_batch.size(0)
            total_mae += torch.abs(outputs - y_batch).sum().item()
    n = len(dataloader.dataset)
    return total_loss / n, total_mae / n

In [ ]:
# 训练模型
EPOCHS = 50

# 记录训练历史
history = {
    'loss': [],
    'val_loss': [],
    'mean_absolute_error': [],
    'val_mean_absolute_error': []
}

for epoch in range(EPOCHS):
    # 训练一个epoch
    train_loss = train_epoch(model, train_loader, criterion, optimizer, device)

    # 在验证集上评估
    val_loss, val_mae = evaluate(model, valid_loader, criterion, device)

    # 计算训练集MAE
    _, train_mae = evaluate(model, train_loader, criterion, device)

    # 记录历史
    history['loss'].append(train_loss)
    history['val_loss'].append(val_loss)
    history['mean_absolute_error'].append(train_mae)
    history['val_mean_absolute_error'].append(val_mae)

    # 每10个epoch打印一次
    if (epoch + 1) % 10 == 0:
        print(f"Epoch {epoch+1:3d}/{EPOCHS} - "
              f"loss: {train_loss:.4f} - "
              f"mae: {train_mae:.4f} - "
              f"val_loss: {val_loss:.4f} - "
              f"val_mae: {val_mae:.4f}")

print("\n训练完成!")

## 7. 可视化训练过程

In [ ]:
def plot_learning_curves(history, metrics=None):
    """
    绘制学习曲线
    Plot learning curves from training history.

    Parameters:
    -----------
    history : dict
        训练历史字典 / Training history dictionary
    metrics : list, optional
        要绘制的指标列表 / List of metrics to plot
    """
    if metrics is None:
        metrics = ['loss']
    n_metrics = len(metrics)
    fig, axes = plt.subplots(1, n_metrics, figsize=(6*n_metrics, 4))

    if n_metrics == 1:
        axes = [axes]

    for ax, metric in zip(axes, metrics):
        ax.plot(history[metric], label=f'Training {metric}')
        ax.plot(history[f'val_{metric}'], label=f'Validation {metric}')
        ax.set_xlabel('Epoch')
        ax.set_ylabel(metric.upper())
        ax.set_title(f'Learning Curve - {metric.upper()}')
        ax.legend()
        ax.grid(True, alpha=0.3)

    plt.tight_layout()
    plt.show()

# 绘制损失曲线和MAE曲线
plot_learning_curves(history, metrics=['loss', 'mean_absolute_error'])

In [ ]:
# 使用pandas绘制（简洁方式）
pd.DataFrame(history).plot(figsize=(10, 5))
plt.grid(True)
plt.gca().set_ylim(0, 1.5)
plt.xlabel('Epoch')
plt.ylabel('Metric Value')
plt.title('Training History')
plt.show()

## 8. 模型评估

In [ ]:
# 在测试集上评估模型
test_loss, test_mae = evaluate(model, test_loader, criterion, device)

print("测试集评估结果:")
print(f"MSE (均方误差): {test_loss:.4f}")
print(f"RMSE (均方根误差): {np.sqrt(test_loss):.4f}")
print(f"MAE (平均绝对误差): {test_mae:.4f}")
print(f"\n解释: 模型预测房价的平均误差约为 ${test_mae*100000:.0f}")

## 9. 使用模型进行预测

In [ ]:
# 对新样本进行预测
model.eval()
X_new = X_test_t[:5].to(device)

with torch.no_grad():
    y_pred = model(X_new)

y_pred_np = y_pred.cpu().numpy()

print("预测结果对比:")
print("="*40)
for i in range(len(X_new)):
    actual = y_test[i]
    predicted = y_pred_np[i][0]
    error = abs(actual - predicted)
    print(f"样本{i+1}: 实际={actual:.3f}, 预测={predicted:.3f}, 误差={error:.3f}")

In [ ]:
# 可视化预测结果
model.eval()
with torch.no_grad():
    y_pred_all = model(X_test_t.to(device)).cpu().numpy().flatten()

plt.figure(figsize=(8, 6))
plt.scatter(y_test, y_pred_all, alpha=0.5, s=10)
plt.plot([0, 5], [0, 5], 'r--', label='Perfect Prediction')
plt.xlabel('Actual House Value (100k $)')
plt.ylabel('Predicted House Value (100k $)')
plt.title('Actual vs Predicted House Values')
plt.legend()
plt.grid(True, alpha=0.3)
plt.axis('equal')
plt.xlim(0, 5)
plt.ylim(0, 5)
plt.tight_layout()
plt.show()

## 小结

### 回归任务的关键点

1. **数据预处理**: 特征标准化对神经网络至关重要
2. **输出层设计**: 回归任务使用单个神经元，无激活函数（线性输出）
3. **损失函数**: 通常使用MSE或MAE
4. **评估指标**: RMSE和MAE更具可解释性

### PyTorch nn.Sequential适用场景

- 层的简单线性堆叠
- 单输入单输出模型
- 快速原型开发

### 后续改进方向

1. 增加隐藏层数量和神经元
2. 添加正则化（Dropout、L2）
3. 使用更优的优化器（Adam）
4. 应用早停和学习率调度

## TF vs PyTorch 对照

| 概念 | TensorFlow / Keras | PyTorch |
|------|-------------------|---------|
| 随机种子 | `tf.random.set_seed(42)` | `torch.manual_seed(42)` |
| 模型构建 | `keras.Sequential([Dense(...), Dense(...)])` | `nn.Sequential(nn.Linear(...), nn.ReLU(), nn.Linear(...))` |
| 全连接层 | `keras.layers.Dense(units, activation)` | `nn.Linear(in_features, out_features)` + `nn.ReLU()` |
| 激活函数 | `Dense(30, activation='relu')` 内置 | `nn.ReLU()` 作为独立层 |
| 编译模型 | `model.compile(loss, optimizer, metrics)` | 分别定义 `criterion` 和 `optimizer` |
| 训练 | `model.fit(X, y, epochs, batch_size, validation_data)` | 手动编写训练循环 + `DataLoader` |
| 评估 | `model.evaluate(X, y)` | 手动编写 `evaluate()` 函数 |
| 预测 | `model.predict(X_new)` | `model(X_new)` (需 `model.eval()` + `torch.no_grad()`) |
| 数据批次 | `batch_size` 参数传入 `fit()` | `DataLoader(dataset, batch_size=32)` |
| 梯度清零 | 自动 | `optimizer.zero_grad()` |
| 反向传播 | 自动 | `loss.backward()` |
| 参数更新 | 自动 | `optimizer.step()` |
| 推理模式 | 自动切换 | `model.eval()` + `torch.no_grad()` |
| 训练模式 | 自动切换 | `model.train()` |
| 模型摘要 | `model.summary()` | `print(model)` 或 `torchinfo.summary()` |

## 练习

### 练习1：添加更多隐藏层

修改模型架构，添加第二个隐藏层（例如30个神经元），观察训练效果的变化：
```python
model = nn.Sequential(
    nn.Linear(8, 30),
    nn.ReLU(),
    nn.Linear(30, 30),   # 新增隐藏层
    nn.ReLU(),
    nn.Linear(30, 1)
).to(device)
```
思考：更深的网络是否一定更好？验证集损失如何变化？

### 练习2：使用Adam优化器

将优化器从SGD替换为Adam，对比训练速度和最终性能：
```python
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
```
思考：Adam和SGD在收敛速度和最终精度上有什么差异？

### 练习3：实现早停(Early Stopping)

Keras中可以使用 `EarlyStopping` 回调，PyTorch中需要手动实现。
请在训练循环中添加早停逻辑：当验证集损失连续N个epoch不再下降时停止训练。
```python
patience = 5
best_val_loss = float('inf')
counter = 0
for epoch in range(EPOCHS):
    train_loss = train_epoch(model, train_loader, criterion, optimizer, device)
    val_loss, val_mae = evaluate(model, valid_loader, criterion, device)
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        counter = 0
        # 保存最佳模型
        best_model_state = model.state_dict()
    else:
        counter += 1
        if counter >= patience:
            print(f'Early stopping at epoch {epoch+1}')
            break
```
思考：早停如何防止过拟合？`patience` 参数如何选择？